# Statistical Inference for the Simple Linear Regression Model — Solutions
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW


In [ ]:
!pip install yfinance statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Solution 1 — $R^2$ by Hand


In [ ]:
TSS = 0.4815
RSS = 0.2066
n   = 1260

ESS = TSS - RSS
R2  = ESS / TSS
R2_check = 1 - RSS / TSS
sigma2_hat = RSS / (n - 2)

print(f'ESS = TSS − RSS = {TSS} − {RSS} = {ESS:.4f}')
print(f'R²  = ESS / TSS = {ESS:.4f} / {TSS:.4f} = {R2:.4f}')
print(f'R²  = 1 − RSS/TSS = 1 − {RSS:.4f}/{TSS:.4f} = {R2_check:.4f}   ← same number')
print(f'sigma²_hat = RSS / (n − 2) = {RSS:.4f} / {n - 2} = {sigma2_hat:.4e}')

**Answers:**
1. About **57%** of the variation in $y$ is explained by the regression (so $R^2 \approx 0.57$). Equivalently, 43% is unexplained — that share comes from factors outside the model.
2. We subtract 2 because we estimated **two parameters** ($\hat{\beta}_0$ and $\hat{\beta}_1$) from the data. Each estimated parameter consumes one degree of freedom. Dividing by $n - 2$ (not $n$) makes $\hat{\sigma}^2$ an **unbiased** estimator of the true error variance $\sigma^2$. Without the correction we systematically under-estimate the variance.
3. With $R^2 = 0.95$, the stock is almost entirely **systematic-risk** driven (only 5% is firm-specific). It behaves like a leveraged or direct play on the market — a classic profile for a mega-cap index constituent or sector ETF.


---
# Solution 2 — Manual SE of $\hat{\beta}_1$


In [ ]:
x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
y = np.array([-1.5, -0.5, 0.2, 1.1, 1.7])
n = len(x)

x_bar = x.mean(); y_bar = y.mean()
num   = ((x - x_bar) * (y - y_bar)).sum()
Sxx   = ((x - x_bar) ** 2).sum()
beta1_hat = num / Sxx
beta0_hat = y_bar - beta1_hat * x_bar

yhat  = beta0_hat + beta1_hat * x
uhat  = y - yhat
RSS   = (uhat ** 2).sum()
sigma2_hat = RSS / (n - 2)
se_beta1 = np.sqrt(sigma2_hat / Sxx)

print(f'x_bar = {x_bar},  y_bar = {y_bar}')
print(f'β1_hat = Σ(x−x_bar)(y−y_bar) / Σ(x−x_bar)² = {num:.4f} / {Sxx:.4f} = {beta1_hat:.4f}')
print(f'β0_hat = y_bar − β1_hat · x_bar = {beta0_hat:.4f}')
print(f'RSS = Σ uhat² = {RSS:.6f}')
print(f'sigma²_hat = RSS / (n−2) = {sigma2_hat:.6f}')
print(f'SE(β1_hat) = sqrt(sigma²_hat / Sxx) = {se_beta1:.4f}')

# Verify with statsmodels
m = sm.OLS(y, sm.add_constant(x)).fit()
print(f'\nstatsmodels: β1_hat = {m.params[1]:.4f},  SE(β1_hat) = {m.bse[1]:.4f}')
print(f'Match? {np.allclose([beta1_hat, se_beta1], [m.params[1], m.bse[1]])}   ← ✓')

**Answers:**
1. $\hat{\beta}_1 = 8.0/10.0 = 0.80$ and $\widehat{\mathrm{SE}}(\hat{\beta}_1) = 0.0365$. Both numbers match `statsmodels` to floating-point precision — the hand formula and the library implement the same algebra.
2. $\widehat{\mathrm{SE}} = 0.0365$ is **small relative to $\hat{\beta}_1 = 0.80$** (about 4.6% of the point estimate). This implies the slope is estimated very precisely — even with only 5 observations, the data are highly informative because $x$ varies cleanly from $-2$ to $+2$ and $y$ tracks it tightly.
3. With 50 observations, the SE would be **smaller** (roughly by a factor of $\sqrt{50/5} \approx 3.16$, all else equal). The SE shrinks as $\sqrt{n}$ grows because $\sum (x - \bar{x})^2$ in the denominator grows linearly with $n$, while $\hat{\sigma}^2$ in the numerator is roughly stable.


---
# Solution 3 — t-Statistic from a Published Coefficient


In [ ]:
beta1_hat = 0.78
se_beta1  = 0.04
n         = 750
beta1_H0  = 1
df        = n - 2

t_obs   = (beta1_hat - beta1_H0) / se_beta1
t_crit  = stats.t.ppf(0.975, df)
p_value = 2 * (1 - stats.t.cdf(abs(t_obs), df))

print(f'H0: β1 = 1   vs.   H1: β1 ≠ 1')
print(f't_obs = ({beta1_hat} − {beta1_H0}) / {se_beta1} = {t_obs:.2f}')
print(f't_crit (α=5%, df={df}) = ±{t_crit:.3f}')
print(f'p-value = 2 · P(T_{df} > |{t_obs:.2f}|) = {p_value:.4e}')

if abs(t_obs) > t_crit:
    print(f'\n|t_obs| = {abs(t_obs):.2f} > {t_crit:.3f} = t_crit   →   REJECT H0')
    print('→ Nestlé is statistically less volatile than the SMI — its beta is below 1 at 5% significance.')
else:
    print('\nDo not reject H0.')

**Answers:**
1. $t = (0.78 - 1) / 0.04 = -5.5$ and $p$-value $\approx 5 \times 10^{-8}$.
2. Yes — the sign $\hat{\beta}_1 - 1 = -0.22$ is negative and the magnitude is huge in $t$-units (5.5 standard errors below 1). Nestlé is **statistically significantly less volatile** than the SMI.
3. The colleague's claim is empirically rejected. With this data we are essentially certain that Nestlé's true beta is below 1 — a defensive consumer-staples profile, exactly as expected for a low-volatility name in food and beverages.


---
# Solution 4 — Full CAPM Regression on a Swiss Stock


In [ ]:
STOCK = 'NESN.SW'
INDEX = '^SSMI'

px = yf.download([STOCK, INDEX], start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna()

y = ret[STOCK]
X = sm.add_constant(ret[INDEX])
model = sm.OLS(y, X).fit()

beta0_hat = model.params['const']
beta1_hat = model.params[INDEX]
se_beta1  = model.bse[INDEX]
t_beta1   = model.tvalues[INDEX]
p_beta1   = model.pvalues[INDEX]
ci_lo, ci_hi = model.conf_int().loc[INDEX]

print(f'Stock: {STOCK} vs {INDEX}, n = {int(model.nobs)} days')
print(f'β0_hat   (intercept) = {beta0_hat:.6f}')
print(f'β1_hat   (CAPM beta) = {beta1_hat:.4f}')
print(f'SE(β1_hat)           = {se_beta1:.4f}')
print(f't-statistic (β1 = 0) = {t_beta1:.2f}')
print(f'p-value     (β1 = 0) = {p_beta1:.3e}')
print(f'R²                   = {model.rsquared:.4f}')
print(f'95% CI for β1_hat    = [{ci_lo:.4f}, {ci_hi:.4f}]')

# Test H0: β1 = 1
test = model.t_test(f'{INDEX} = 1')
print(f'\nTest H0: β1 = 1')
print(f'  t = {float(np.squeeze(test.tvalue)):.2f}')
print(f'  p = {float(np.squeeze(test.pvalue)):.4e}')

**Answers:** (numbers are for Nestlé NESN.SW)
1. $\hat{\beta}_1 \approx 0.7$ to $0.8$ depending on the period. Nestlé is **defensive** ($\beta_1 < 1$) — consistent with the consumer-staples sector.
2. Yes — $p$-value for $\beta_1 = 0$ is essentially zero. Nestlé clearly moves with the broad market.
3. Yes — typically $|t|$ for $\beta_1 = 1$ is well above 5, so we reject. Nestlé is **significantly less volatile** than the SMI.
4. The 95% CI sits entirely below 1, so it contains neither 0 nor 1. Both null hypotheses are rejected at 5%. The CI gives a range of plausible betas, and that range is bounded comfortably below the market beta of 1.


---
# Solution 5 — Confidence Interval as a Range


In [ ]:
df = int(model.df_resid)
t_crit = stats.t.ppf(0.975, df)

ci_lo_manual = beta1_hat - t_crit * se_beta1
ci_hi_manual = beta1_hat + t_crit * se_beta1

print(f't_crit (α=5%, df={df}) = {t_crit:.4f}')
print(f'CI (manual)     = [{ci_lo_manual:.4f}, {ci_hi_manual:.4f}]')
print(f'CI (statsmodels) = [{ci_lo:.4f}, {ci_hi:.4f}]')
print(f'Match? {np.allclose([ci_lo_manual, ci_hi_manual], [ci_lo, ci_hi])}   ← ✓')

# Forest plot
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.errorbar([beta1_hat], [0],
            xerr=[[beta1_hat - ci_lo_manual], [ci_hi_manual - beta1_hat]],
            fmt='o', color=RED, ecolor='black', capsize=10, lw=2.5, ms=14)
ax.axvline(0, color=GREY, ls='--', lw=1.2)
ax.axvline(1, color=ORANGE, ls=':', lw=1.5)
ax.text(0, 0.4, r'$\beta_1 = 0$  (no effect)', ha='center', fontsize=10, color=GREY)
ax.text(1, 0.4, r'$\beta_1 = 1$  (market β)', ha='center', fontsize=10, color=ORANGE)
ax.text(beta1_hat, -0.25, fr'$\hat{{\beta}}_1 = {beta1_hat:.3f}$',
        ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(-0.6, 0.7); ax.set_yticks([])
ax.set_xlim(-0.15, max(1.2, ci_hi_manual + 0.1))
ax.set_xlabel(r'$\hat{\beta}_1$')
ax.set_title(f"95% confidence interval for {STOCK}'s CAPM beta",
             fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

**Answers:**
1. The 95% CI is the manual range from the code above. Whether 0 and 1 fall inside depends on the stock. For Nestlé: 0 is outside (so $\beta_1 \neq 0$), 1 is outside (so $\beta_1 \neq 1$) — exactly what the $t$-tests said.
2. The 95% CI contains exactly those null values $\beta_{1,H_0}$ for which $|t_{\text{obs}}|$ would NOT exceed $t_{\text{crit}}$ at the 5% level — i.e. the CI is the set of nulls that we cannot reject at 5%. CI and $t$-test are equivalent decision rules.
3. A 99% CI would be **wider**, because we use a larger critical value ($t_{0.005,\,n-2} \approx 2.58$ vs. $t_{0.025,\,n-2} \approx 1.96$). Higher confidence requires a wider interval — there is no free lunch.


---
# Solution 6 — Four-Step Test on TLT


In [ ]:
# sort=True is not cosmetic. pandas 3 still sorts a DatetimeIndex concat by
# default but warns that pandas 4 will not; an unsorted index then breaks every
# downstream .loc['2020-01-01':'2020-06-30'] date slice. Say what we mean.
tlt = yf.download('TLT', start='2019-01-01', end='2024-12-31',
                  auto_adjust=True, progress=False)['Close']
tnx = yf.download('^TNX', start='2019-01-01', end='2024-12-31',
                  auto_adjust=True, progress=False)['Close']

tlt_ret = tlt.pct_change()
tnx_diff = tnx.diff()
data = pd.concat([tlt_ret, tnx_diff], axis=1, sort=True).dropna()
data.columns = ['TLT', 'dTNX']

y = data['TLT']
X = sm.add_constant(data['dTNX'])
model = sm.OLS(y, X).fit()

beta1_hat = model.params['dTNX']
se_beta1  = model.bse['dTNX']
df_       = int(model.df_resid)
t_crit    = stats.t.ppf(0.975, df_)
D_H0      = 17.0            # modified duration TLT publishes, in years
beta1_H0  = -D_H0 / 100     # ^TNX is in percentage points, TLT returns are fractions
t_obs     = (beta1_hat - beta1_H0) / se_beta1
p_val     = 2 * (1 - stats.t.cdf(abs(t_obs), df_))

print(f'=== Four-step test: H0: β1 = {beta1_H0:.2f}  (modified duration {D_H0:.0f} years) ===')
print(f'\nStep 1.  H0: β1 = {beta1_H0:.2f}   vs.   H1: β1 ≠ {beta1_H0:.2f}')
print(f'\nStep 2.  α = 5%, two-sided → t_crit = ±{t_crit:.3f}')
print(f'\nStep 3.  β1_hat = {beta1_hat:.4f},  SE = {se_beta1:.4f}')
print(f'         t = ({beta1_hat:.4f} − ({beta1_H0:.2f})) / {se_beta1:.4f} = {t_obs:.2f}')
print(f'         p-value = {p_val:.3e}')
if abs(t_obs) > t_crit:
    print(f'\nStep 4.  |t| = {abs(t_obs):.2f} > {t_crit:.3f}   →   REJECT H0')
else:
    print(f'\nStep 4.  |t| = {abs(t_obs):.2f} < {t_crit:.3f}   →   DO NOT REJECT H0')

implied_duration = -beta1_hat * 100   # ^TNX is in percent, so coef × 100 ≈ duration in years
print(f'\nImplied duration: −β1_hat × 100 = {implied_duration:.1f} years')
print(f'Published TLT duration: ~17 years')

# Verify with built-in
print(f'\nVerify: model.t_test(\'dTNX = -0.17\')')
print(model.t_test('dTNX = -0.17'))

**Answers:**
1. Read the $t$-statistic the cell prints. It is $t = (\hat{\beta}_1 - (-0.17)) / \widehat{\mathrm{SE}}(\hat{\beta}_1)$, so it measures how far the sample slope sits from the slope a 17-year duration implies, in standard errors.
2. The implied duration is $-\hat{\beta}_1 \times 100$ years. Over 2019–2024 this regression lands **below the published 17 years** — the same result the M4 solution reports, where the identical regression is run with returns in percent so that the slope *is* the duration in years. Whether $H_0$ is formally rejected depends on the sample: with roughly 1,500 daily observations the standard error is small, so even a difference of one or two duration years shows up as a significant $t$.
3. Yes, consistent. And this is the classic gap between statistical and economic significance: rejecting $H_0: \beta_1 = -0.17$ at 5% does not mean the published duration is wrong. The regression-implied duration is a noisy proxy that depends on the yield series chosen (^TNX is the 10-year point, while TLT holds 20+ year bonds), on the sample period, and on the assumption that the whole curve shifts in parallel. The published duration is computed directly from the fund's cash flows and is the cleaner measure; the regression is the market-implied cross-check.


---
# Solution 7 — Heteroskedasticity Detection


In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

# Reuse the TLT model from Solution 6
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.scatter(model.fittedvalues, model.resid, s=8, alpha=0.4, color=GREY)
ax.axhline(0, color=RED, lw=1.5)
ax.set_xlabel('Fitted Y_hat'); ax.set_ylabel('Residual u_hat')
ax.set_title('A.  Residuals vs fitted — look for fanning',
             fontweight='bold', loc='left')

ax = axes[1]
ax.plot(model.resid.index, model.resid.values, color=GREY, lw=0.6)
ax.axhline(0, color=RED, lw=1.5)
ax.set_xlabel('Date'); ax.set_ylabel('Residual u_hat')
ax.set_title('B.  Residuals over time — look for volatility clusters',
             fontweight='bold', loc='left')

plt.tight_layout(); plt.show()

# Breusch-Pagan test
lm, lm_p, _, _ = het_breuschpagan(model.resid, model.model.exog)
print(f'Breusch-Pagan test:')
print(f'  H0: homoskedasticity   vs.   H1: heteroskedasticity')
print(f'  LM statistic = {lm:.2f}')
print(f'  LM p-value   = {lm_p:.4e}')
if lm_p < 0.05:
    print('  → Reject H0: heteroskedasticity DETECTED. Use HC1 robust standard errors.')
else:
    print('  → Do not reject H0: classical SE looks fine.')

**Answers:**
1. The residuals-vs-fitted plot for financial returns typically shows fanning — particularly during stress episodes (March 2020 COVID crash, 2022 inflation/rate shock). Residuals over time show **volatility clusters** — large moves bunched together. Both signatures point to heteroskedasticity.
2. The Breusch-Pagan $p$-value is essentially zero for almost any financial-returns regression on this sample. We **reject homoskedasticity** at any reasonable level.
3. Classical OLS standard errors are **biased downward** under heteroskedasticity — they make our $t$-statistics look larger than they should be, so we **over-reject** $H_0$. The result: we claim "significant" relationships that are not actually significant once we use correctly-sized standard errors. This is why HC1 is the default in modern empirical work.


---
# Solution 8 — OLS vs. HC1 Robust SE


In [ ]:
# Re-use the model from Solution 4 (or replace with any other regression)
# Note: model variables get reassigned in Solution 6 — refit Solution 4's setup here:

px = yf.download(['NESN.SW', '^SSMI'], start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna()
y = ret['NESN.SW']
X = sm.add_constant(ret['^SSMI'])

model_ols = sm.OLS(y, X).fit()
model_hc  = sm.OLS(y, X).fit(cov_type='HC1')

# Build a clean comparison table
def line(label, ols, hc):
    print(f'{label:<22} {ols:>14}    {hc:>14}')

print('Comparison: OLS vs. HC1 robust standard errors\n')
print(f'{"":>22} {"OLS":>14}    {"HC1":>14}')
print('-' * 60)
for k in ['const', '^SSMI']:
    print(f'\n{k}:')
    line('  β_hat',         f'{model_ols.params[k]:.4f}',  f'{model_hc.params[k]:.4f}')
    line('  SE',            f'{model_ols.bse[k]:.4f}',     f'{model_hc.bse[k]:.4f}')
    line('  t-stat',        f'{model_ols.tvalues[k]:.2f}', f'{model_hc.tvalues[k]:.2f}')
    line('  p-value',       f'{model_ols.pvalues[k]:.3e}', f'{model_hc.pvalues[k]:.3e}')

se_ols = model_ols.bse['^SSMI']
se_hc  = model_hc.bse['^SSMI']
pct = (se_hc - se_ols) / se_ols * 100
print(f'\n→ Slope SE changed by {pct:+.1f}% when switching to HC1.')

**Answers:**
1. The slope SE typically grows by 5–15% when switching to HC1 on Swiss-blue-chip data — moderate but non-trivial. For more volatile names (e.g. tech, biotech) or stress periods, the change can be 30%+.
2. The point estimate $\hat{\beta}_1$ does **not** change at all. It is the same OLS minimisation — only the standard error formula changes. This is a critical point: HC corrects the inference, not the estimate.
3. Always report HC1 (or HC3 for very small samples) when (a) you cannot rule out heteroskedasticity a priori — which in finance is almost always — or (b) the Breusch-Pagan test rejects homoskedasticity. In modern empirical finance papers, "robust standard errors in parentheses" is the default — classical SEs are now the exception, not the rule.


---
# Solution 9 — Apple's CAPM Beta Pre- vs. Post-COVID


In [ ]:
data = yf.download(['AAPL', '^GSPC'], start='2018-01-01', end='2024-12-31',
                   auto_adjust=True, progress=False)['Close']
ret = data.pct_change().dropna()
# rename by label, never by position: yfinance orders the Close columns alphabetically
ret = ret.rename(columns={'^GSPC': 'SP500'})[['AAPL', 'SP500']]

# Split into pre- and post-COVID
pre  = ret.loc['2018-01-01':'2019-12-31']
post = ret.loc['2020-01-01':'2024-12-31']

def fit_capm(returns):
    y = returns['AAPL']
    X = sm.add_constant(returns['SP500'])
    m = sm.OLS(y, X).fit(cov_type='HC1')
    return m.params['SP500'], m.bse['SP500'], int(m.nobs)

b_pre,  se_pre,  n_pre  = fit_capm(pre)
b_post, se_post, n_post = fit_capm(post)

print(f'Pre-COVID  (2018-2019):  β1_hat = {b_pre:.4f}   SE = {se_pre:.4f}   n = {n_pre}')
print(f'Post-COVID (2020-2024):  β1_hat = {b_post:.4f}  SE = {se_post:.4f}   n = {n_post}')

# Test of difference (approximate two-sample z-test)
diff   = b_post - b_pre
se_dif = np.sqrt(se_pre**2 + se_post**2)
z      = diff / se_dif
p      = 2 * (1 - stats.norm.cdf(abs(z)))

print(f'\nDifference: β_post − β_pre = {diff:.4f}')
print(f'SE of difference: sqrt(SE_pre² + SE_post²) = {se_dif:.4f}')
print(f'z = {z:.2f}, two-sided p = {p:.4f}')

if p < 0.05:
    print('\n→ Apple\'s beta CHANGED significantly between the two periods.')
else:
    print('\n→ Cannot reject that Apple\'s beta is the same in both periods.')

# A more rigorous test: pooled regression with a post-COVID interaction dummy
ret_join = ret.copy()
ret_join['POST'] = (ret_join.index >= '2020-01-01').astype(float)
ret_join['SP500_x_POST'] = ret_join['SP500'] * ret_join['POST']
X_join = sm.add_constant(ret_join[['SP500', 'POST', 'SP500_x_POST']])
m_join = sm.OLS(ret_join['AAPL'], X_join).fit(cov_type='HC1')
print('\n--- Rigorous test (pooled regression with interaction) ---')
print(f'Interaction coefficient β_SP500×POST = {m_join.params["SP500_x_POST"]:.4f}')
print(f'  SE  = {m_join.bse["SP500_x_POST"]:.4f}')
print(f'  t   = {m_join.tvalues["SP500_x_POST"]:.2f}')
print(f'  p   = {m_join.pvalues["SP500_x_POST"]:.4f}')

**Answers:**
1. Read the two betas the cell prints. Do not commit to a range in advance: the pre-COVID window is short and contains the Q4-2018 selloff, so the estimate is sensitive to exactly where the sample starts, and the sign of the shift can go either way.
2. Decide from the output: compare the approximate $z$-test with the pooled regression that carries the interaction term. The two should agree; if they do not, the interaction regression is the more rigorous of the two because it uses one pooled error variance.
3. Possible explanations: (a) Apple grew to ~7% of the S&P 500 by 2023, so the index now partly co-moves with Apple — the slope captures that increased mechanical link; (b) the tech-heavy COVID rally made Apple more correlated with the broad market; (c) higher leverage in retail investor positioning amplified Apple's market response. The takeaway: betas are **not stable** through time, and forward-looking risk management requires rolling re-estimation.


---
# Solution 10 — Build Your Own Inference Study (example)


In [ ]:
# Hypothesis: TSLA has a CAPM beta significantly above 2 — i.e. it is at least
# twice as volatile as the S&P 500.

px = yf.download(['TSLA', '^GSPC'], start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna()
# rename by label, never by position: yfinance orders the Close columns alphabetically
ret = ret.rename(columns={'^GSPC': 'SP500'})[['TSLA', 'SP500']]

X = sm.add_constant(ret['SP500'])
y = ret['TSLA']
model = sm.OLS(y, X).fit(cov_type='HC1')

print(model.summary())
print('\nTest H0: β1 = 2  vs.  H1: β1 ≠ 2 (HC1 robust):')
print(model.t_test('SP500 = 2'))

**Executive summary:**
Tesla's estimated CAPM beta against the S&P 500 over 2019-2024 is the $\hat{\beta}_1$ printed above, with the robust HC1 standard error next to it. State whether the point estimate is significantly above 2 using the $t$-statistic and $p$-value the cell reports, and quote the 95% confidence interval as printed — the write-up must carry your numbers, not remembered ones. Tesla is a high-volatility name, far more sensitive to broad-market moves than the average S&P 500 constituent. The $R^2$ is nevertheless low, so the majority of Tesla's variance is **idiosyncratic** (Musk tweets, EV demand shocks, China factory news), not systematic. A risk manager would size a Tesla position based on this beta but layer additional hedges or limits for the firm-specific tail risk.


---
# 🔥 Challenge Solution — The "Low-Beta Anomaly"


In [ ]:
tickers = ['KO', 'JNJ', 'AAPL', 'TSLA', 'NVDA']
px = yf.download(tickers + ['^GSPC'], start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna() * 100   # in percent
mkt = ret['^GSPC']

results = {}
for t in tickers:
    X = sm.add_constant(mkt)
    m = sm.OLS(ret[t], X).fit()
    results[t] = {
        'beta1': m.params['^GSPC'],
        'se':    m.bse['^GSPC'],
        'mean_annual_return': ret[t].mean() * 252,  # daily % × 252, RAW (not excess)
    }

df_res = pd.DataFrame(results).T
print('Beta and mean annual return for each ticker:')
print(df_res.round(3))

# Plot mean return vs beta
fig, ax = plt.subplots(figsize=(9, 5.5))
betas = df_res['beta1'].values
mr    = df_res['mean_annual_return'].values

ax.scatter(betas, mr, s=180, color=YELLOW, edgecolor='black', linewidth=1, zorder=5)
for t, b, r in zip(df_res.index, betas, mr):
    ax.annotate(t, (b, r), textcoords='offset points', xytext=(10, 8), fontsize=11,
                fontweight='bold')

# CAPM-implied line — passes through origin with slope = mean(SP500 annual return)
mkt_annual = mkt.mean() * 252
xx = np.linspace(0.4, max(betas)+0.3, 50)
ax.plot(xx, xx * mkt_annual, color=GREY, ls='--', lw=1.5,
        label=f'CAPM-implied line, r_f = 0 (slope = S&P 500 avg = {mkt_annual:.1f}%)')
# The security market line runs through the origin only in EXCESS returns. These are
# raw returns, so the dashed line carries the implicit assumption r_f = 0.

# Empirical line through the dots
slope_emp, intercept_emp = np.polyfit(betas, mr, 1)
ax.plot(xx, intercept_emp + slope_emp * xx, color=RED, lw=2.5,
        label=f'Empirical fit:  slope = {slope_emp:.1f}%/unit β')

ax.axhline(0, color=GREY, lw=0.6)
ax.set_xlabel(r'CAPM β1_hat  (against S&P 500)')
ax.set_ylabel('Mean annual return (%)')
ax.set_title('The low-beta anomaly: empirical slope is often FLATTER than CAPM predicts',
             fontweight='bold', loc='left')
ax.legend(loc='upper left', frameon=False)
plt.tight_layout(); plt.show()

**What you see:** For a long sample of S&P 500 stocks, the empirically estimated risk-return line is much **flatter** than CAPM predicts — sometimes even slightly negative. Low-beta stocks earned roughly as much as high-beta stocks, contradicting the textbook CAPM. This anomaly is robust across countries, decades, and asset classes (Frazzini-Pedersen 2014, "Betting Against Beta", JFE). It is widely interpreted as a sign that real-world investors are leverage-constrained: they cannot lever up safe assets, so they tilt their portfolios toward riskier names instead, pushing high-beta prices up (and forward returns down).

In our 5-stock cherry-picked sample (mostly mega-cap winners) you might see the CAPM relation hold — the anomaly is most visible in broad cross-sections of hundreds of stocks. The exercise illustrates the spirit of the test, not the formal result.

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*
